# 02 — Modélisation : prédiction du niveau de risque

Ce notebook prépare les variables (sans fuite de données), entraîne et compare trois modèles (Régression logistique, Random Forest, XGBoost), puis analyse les métriques et l'importance des variables.

> Pour reproduire hors notebook, voir `src/preprocessing.py`, `src/train.py` et `src/predict.py`.


## 2. Préparation et prévention de la fuite de données

In [ ]:
# Transformation simple de la date, puis définition de la cible et des variables explicatives.
df_model = df.copy()
df_model["date"] = pd.to_datetime(df_model["date"], errors="coerce")
df_model["jour_annee"] = df_model["date"].dt.dayofyear

TARGET = "niveau_risque"
# Ces champs sont exclus : identifiant, date brute, constante annuelle et résultats sanitaires
# directement liés à la cible (fuite de données).
excluded_columns = {
    "id_observation", "date", "annee", TARGET,
    "cas_paludisme_simules", "taux_incidence_simule_pour_1000",
}
feature_columns = [column for column in df_model.columns if column not in excluded_columns]
X = df_model[feature_columns].copy()

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df_model[TARGET].astype(str))
class_names = label_encoder.classes_

categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()

print("Variables catégorielles :", categorical_features)
print("Variables numériques :", numeric_features)
print("Classes :", list(class_names))
print("Nombre de variables avant encodage :", X.shape[1])
    

In [ ]:
# Séparation stratifiée : le test reste isolé avant tout apprentissage du prétraitement.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Train :", X_train.shape, "Test :", X_test.shape)
print("Répartition train :", dict(zip(*np.unique(y_train, return_counts=True))))
print("Répartition test :", dict(zip(*np.unique(y_test, return_counts=True))))
    

In [ ]:
# Encodage One-Hot des variables catégorielles et standardisation des variables numériques.
# La compatibilité sparse_output / sparse couvre plusieurs versions de scikit-learn.
try:
    one_hot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    one_hot = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), numeric_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", one_hot),
        ]), categorical_features),
    ],
    remainder="drop",
)
print("Préprocesseur créé : imputation + normalisation + One-Hot Encoding.")
    

## 3. Modélisation : trois algorithmes comparés

In [ ]:
# XGBoost est requis pour le troisième modèle.
try:
    from xgboost import XGBClassifier
except ImportError as exc:
    raise ImportError(
        "XGBoost est requis. Installez-le avec : %pip install xgboost"
    ) from exc

models = {
    "Régression logistique": Pipeline([
        ("preprocessing", preprocessor),
        ("model", LogisticRegression(max_iter=2000, class_weight="balanced")),
    ]),
    "Random Forest": Pipeline([
        ("preprocessing", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=300,
            max_depth=12,
            min_samples_leaf=2,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ]),
    "XGBoost": Pipeline([
        ("preprocessing", preprocessor),
        ("model", XGBClassifier(
            n_estimators=250,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.85,
            colsample_bytree=0.85,
            objective="multi:softprob",
            eval_metric="mlogloss",
            num_class=len(class_names),
            random_state=RANDOM_STATE,
            n_jobs=2,
        )),
    ]),
}

# Pondération des observations : alternative robuste à SMOTE pour ce CSV mixte.
train_sample_weight = compute_sample_weight(class_weight="balanced", y=y_train)
results = []
predictions = {}
probabilities = {}

for name, model in models.items():
    if name == "XGBoost":
        model.fit(X_train, y_train, model__sample_weight=train_sample_weight)
    else:
        model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)
    predictions[name] = y_pred
    probabilities[name] = y_proba
    results.append({
        "modèle": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision_weighted": precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "rappel_weighted": recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "f1_weighted": f1_score(y_test, y_pred, average="weighted", zero_division=0),
        "roc_auc_ovr_weighted": roc_auc_score(y_test, y_proba, multi_class="ovr", average="weighted"),
    })

results_df = pd.DataFrame(results).set_index("modèle").sort_values("f1_weighted", ascending=False)
display(results_df.style.format("{:.3f}"))
    

## 4. Évaluation détaillée

In [ ]:
best_model_name = results_df.index[0]
print("Meilleur modèle selon le F1 pondéré :", best_model_name)

for name in models:
    print("\n" + "=" * 70)
    print(name)
    print(classification_report(
        y_test,
        predictions[name],
        labels=np.arange(len(class_names)),
        target_names=class_names,
        zero_division=0,
    ))
    cm = confusion_matrix(y_test, predictions[name], labels=np.arange(len(class_names)))
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f"Matrice de confusion — {name}")
    plt.xlabel("Classe prédite")
    plt.ylabel("Classe réelle")
    plt.tight_layout()
    plt.show()
    

In [ ]:
# Comparaison graphique des métriques principales.
metric_columns = ["accuracy", "precision_weighted", "rappel_weighted", "f1_weighted", "roc_auc_ovr_weighted"]
results_df[metric_columns].plot(kind="bar", figsize=(14, 6), ylim=(0, 1.05))
plt.title("Comparaison des modèles")
plt.ylabel("Score")
plt.xlabel("")
plt.xticks(rotation=0)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()
    

## 5. Importance des variables

In [ ]:
# Importance des variables pour les modèles à base d'arbres.
for model_name in ["Random Forest", "XGBoost"]:
    fitted = models[model_name]
    transformed_names = fitted.named_steps["preprocessing"].get_feature_names_out()
    importances = fitted.named_steps["model"].feature_importances_
    importance_df = pd.DataFrame({
        "variable": transformed_names,
        "importance": importances,
    }).sort_values("importance", ascending=False).head(20)

    print(f"Top 20 variables — {model_name}")
    display(importance_df)
    plt.figure(figsize=(10, 7))
    sns.barplot(data=importance_df.sort_values("importance"), x="importance", y="variable", color="#2a9d8f")
    plt.title(f"Variables les plus importantes — {model_name}")
    plt.xlabel("Importance")
    plt.ylabel("")
    plt.tight_layout()
    plt.show()
    

## 6. Conclusion et limites

- Le tableau `results_df` compare les trois modèles demandés.
- Le choix final doit privilégier le **rappel** et le **F1** de la classe `eleve` si l'objectif est de ne pas manquer une période à risque.
- Les variables sanitaires simulées ont été exclues afin d'éviter la fuite de données.
- Le fichier est synthétique : une utilisation opérationnelle nécessiterait des observations validées par les services de santé, les sources OMS/HDX et des historiques météo vérifiés.
- Les scores obtenus sur ce fichier ne doivent pas être interprétés comme une estimation épidémiologique réelle de la Guinée.
    